In [340]:
import pandas as pd
import numpy as np 
from scipy import stats
from scipy.stats import bootstrap as bootstrap
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn import catplot
import random as rd
import os
from sklearn.preprocessing import StandardScaler as ss
from sklearn.decomposition import PCA as pca
import math
from scipy.stats import linregress
import mne
%matplotlib qt
import os
from itertools import compress

import mne
from mne import datasets
import numpy as np
from nilearn.plotting import plot_design_matrix

import mne_nirs
import mne_bids
# Import StatsModels
import statsmodels.formula.api as smf
import matplotlib as mpl

# Import MNE processing
from mne.preprocessing.nirs import beer_lambert_law, optical_density

# Import MNE-BIDS processing
from mne_bids import BIDSPath, get_entity_vals, read_raw_bids,stats

from mne_nirs.channels import get_long_channels, get_short_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.io.fold import fold_channel_specificity

# Import MNE-NIRS processing
from mne_nirs.statistics import run_glm, statsmodels_to_results
from mne_nirs.visualisation import plot_glm_group_topo, plot_glm_surface_projection 
import os.path as op

from mne.datasets.testing import data_path
from mne.viz import set_3d_view

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm

In [342]:
# Import common libraries
from collections import defaultdict
from copy import deepcopy
from itertools import compress
from pprint import pprint

# Import Plotting Library
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import StatsModels
import statsmodels.formula.api as smf
from mne import Epochs, events_from_annotations, set_log_level
from mne.preprocessing.nirs import (
    beer_lambert_law,
    optical_density,
    scalp_coupling_index,
    temporal_derivative_distribution_repair,
)

# Import MNE processing
from mne.viz import plot_compare_evokeds

# Import MNE-BIDS processing
from mne_bids import BIDSPath, read_raw_bids

# Import MNE-NIRS processing
from mne_nirs.channels import get_long_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.signal_enhancement import enhance_negative_correlation

# Set general parameters
set_log_level("WARNING")  # Don't show info, as it is repetitive for many subjects

In [345]:
# root = r'C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v2'
root = r'C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v2\pid002.snirf'

# subjects = [f for f in os.listdir(root) if os.path.isfile(os.path.join(root, f))]

def individual_analysis(root, file_name):
    # Construct the full path to the SNIRF file
    subject_file_path = os.path.join(root, file_name)
    
    # Check if the file exists
    if not os.path.exists(subject_file_path):
        print(f"File not found: {subject_file_path}")
        return None, None

    # Load the raw intensity data from the SNIRF file
    raw_intensity = mne.io.read_raw_snirf(subject_file_path, verbose=False)

    # Preserve the original montage
    original_montage = raw_intensity.get_montage()
    raw_intensity.set_montage(original_montage, match_case=False)  # Ensures the montage is set correctly

    # Modify the annotations as needed
    for i, desc in enumerate(raw_intensity.annotations.description):
        if desc in ['1', '3', '5']:
            raw_intensity.annotations.duration[i] = 36

    # Print the updated annotations to verify
    print(raw_intensity.annotations)
    unwanted = np.nonzero((raw_intensity.annotations.description == "7") |
                          (raw_intensity.annotations.description == "2") |
                          (raw_intensity.annotations.description == "4") |
                          (raw_intensity.annotations.description == "6"))
    raw_intensity.annotations.delete(unwanted)
    raw_intensity.annotations.rename({"1": "1st Order", "3": "1st Order", "5": "2nd Order"})
    event_dict = {'1st Order': 1, '1st Order': 3, "2nd Order": 5}

    # Convert signal to optical density and determine bad channels
    raw_od = optical_density(raw_intensity)
    sci = scalp_coupling_index(raw_od, h_freq=1.35, h_trans_bandwidth=0.1)
    raw_od.info["bads"] = list(compress(raw_od.ch_names, sci < 0.5))
    raw_od.interpolate_bads()
    # Skip the file if more than 10 channels are marked as bad
    if len(raw_od.info["bads"]) > 10:
        print(f"File {file_name} has more than 10 bad channels, skipping.")
        return None, None, None, None

    # Downsample and apply signal cleaning techniques
    raw_od.resample(1.4)
    raw_od = temporal_derivative_distribution_repair(raw_od)

    # Convert to haemoglobin and filter
    raw_haemo = beer_lambert_law(raw_od, ppf=0.1)
    raw_haemo = raw_haemo.filter(
        0.03, 0.3, h_trans_bandwidth=0.1, l_trans_bandwidth=0.01, verbose=False
    )

    # Preserve the montage for haemoglobin data
    raw_haemo.set_montage(original_montage, match_case=False)

    # Apply further data cleaning techniques and extract epochs
    # raw_haemo = enhance_negative_correlation(raw_haemo)

    # Extract events but ignore those with the word "Ends"
    events, event_dict = events_from_annotations(
        raw_haemo, verbose=False, regexp="^(?![Ends]).*$"
    )
    epochs = Epochs(
        raw_haemo,
        events,
        event_id=event_dict,
        tmin=-5,
        tmax=36,
        reject=dict(hbo=200e-6),
        reject_by_annotation=True,
        proj=True,
        baseline=(None, 0),
        detrend=0,
        preload=True,
        verbose=False,
    )

    return raw_haemo, epochs


In [312]:
from collections import defaultdict
import os

all_evokeds = defaultdict(list)

# Define the root directory where the SNIRF files are stored
root = r'C:\Users\pkell\OneDrive\Documents\fNIRS\pilot data\rel_match\v2'

# List all .snirf files in the root directory
snirf_files = [f for f in os.listdir(root) if f.endswith('.snirf')]

# Loop through each SNIRF file in the folder
for file_name in snirf_files:
    # Analyze the data and return the raw haemoglobin and epochs
    raw_haemo, epochs = individual_analysis(root, file_name)

    # Check if analysis returned valid results before proceeding
    if raw_haemo is None or epochs is None:
        print(f"Skipping file {file_name} due to missing or invalid data.")
        continue

    # Save individual-evoked participant data along with others in all_evokeds
    for condition in epochs.event_id:
        # Check if the condition's epochs contain data before averaging
        if len(epochs[condition]) == 0:
            print(f"Skipping empty epoch for condition {condition} in file {file_name}.")
            continue
        all_evokeds[condition].append(epochs[condition].average())


C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:18: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw_intensity.set_montage(original_montage, match_case=False)  # Ensures the montage is set correctly
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:50: RuntimeWarning: Source-detector distances are greater than 10 cm. Large distances will result in invalid data, and are likely due to optode locations being stored in a  unit other than meters.
  raw_haemo = beer_lambert_law(raw_od, ppf=0.1)


<Annotations | 198 segments: 1 (6), 2 (81), 3 (3), 4 (36), 5 (9), 6 (45), ...>


C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:56: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw_haemo.set_montage(original_montage, match_case=False)
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:18: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw_intensity.set_montage(original_montage, match_case=False)  # Ensures the montage is set correctly


<Annotations | 198 segments: 1 (6), 2 (81), 3 (3), 4 (36), 5 (9), 6 (45), ...>


C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:50: RuntimeWarning: Source-detector distances are greater than 10 cm. Large distances will result in invalid data, and are likely due to optode locations being stored in a  unit other than meters.
  raw_haemo = beer_lambert_law(raw_od, ppf=0.1)
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:56: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw_haemo.set_montage(original_montage, match_case=False)
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:18: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw_intensity.set_montage(original_montage, match_case=False)  # Ensures the montage is set correctly


...

C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:50: RuntimeWarning: Source-detector distances are greater than 10 cm. Large distances will result in invalid data, and are likely due to optode locations being stored in a  unit other than meters.
  raw_haemo = beer_lambert_law(raw_od, ppf=0.1)
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:56: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw_haemo.set_montage(original_montage, match_case=False)
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:65: RuntimeWarning: All epochs were dropped!
You might need to alter reject/flat-criteria or drop bad channels to avoid this. You can use Epochs.plot_drop_log() to see which channels are responsible for the dropping of epochs.
  epochs = Epochs(
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:18: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head tran

Skipping empty epoch for condition 1st Order in file pid025.snirf.
Skipping empty epoch for condition 2nd Order in file pid025.snirf.
<Annotations | 198 segments: 1 (6), 2 (81), 3 (3), 4 (36), 5 (9), 6 (45), ...>


C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:50: RuntimeWarning: Source-detector distances are greater than 10 cm. Large distances will result in invalid data, and are likely due to optode locations being stored in a  unit other than meters.
  raw_haemo = beer_lambert_law(raw_od, ppf=0.1)
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:56: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw_haemo.set_montage(original_montage, match_case=False)
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:18: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw_intensity.set_montage(original_montage, match_case=False)  # Ensures the montage is set correctly
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:39: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw_od.interpolate_bads()


<Annotations | 198 segments: 1 (6), 2 (81), 3 (3), 4 (36), 5 (9), 6 (45), ...>


C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:50: RuntimeWarning: Source-detector distances are greater than 10 cm. Large distances will result in invalid data, and are likely due to optode locations being stored in a  unit other than meters.
  raw_haemo = beer_lambert_law(raw_od, ppf=0.1)
C:\Users\pkell\AppData\Local\Temp\ipykernel_24408\3417753752.py:56: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw_haemo.set_montage(original_montage, match_case=False)


In [314]:
 # Define channels in each region of interest
left = [[1, 1], [1, 2], [2, 1], [2, 2], [2, 3], [3, 1], [3, 3], [4, 3], [4, 2], [4, 4], [5, 3], [5, 4],
            [11, 9], [11, 8], [12, 8], [12, 10], [13, 8], [13, 9], [13, 10], [13, 11]]
right = [[6, 5], [6, 6], [7, 5], [7, 6], [7,7], [8, 5], [8,7], [9, 7], [9,6], [10,7], [14, 12], [14, 13], 
             [15, 12], [15, 14], [16, 12], [16, 13], [16, 14],[16,15]]

In [316]:
# Specify the figure size and limits per chromophore
fig, axes = plt.subplots(nrows=1, ncols=len(all_evokeds), figsize=(17, 5))
lims = dict(hbo=[-5, 6], hbr=[-5, 6])

for pick, color in zip(["hbo", "hbr"], ["r", "b"]):
    for idx, evoked in enumerate(all_evokeds):
        plot_compare_evokeds(
            {evoked: all_evokeds[evoked]},
            combine="mean",
            picks=pick,
            axes=axes[idx],
            show=False,
            colors=[color],
            legend=False,
            ylim=lims,
            ci=0.95,
            show_sensors=idx == 2,
        )
        axes[idx].set_title(f"{evoked}")
axes[0].legend(["Oxyhaemoglobin", "Deoxyhaemoglobin"])

In [ ]:
##  NEED TO GET MORE EXACT ROI FROM MONTAGE ON EXP COMPUTER

In [61]:
 # Define channels in each region of interest
left = [[1, 1], [1, 2], [2, 1], [2, 2], [2, 3], [3, 1], [3, 3], [4, 3], [4, 2], [4, 4], [5, 3], [5, 4],
            [11, 9], [11, 8], [12, 8], [12, 10], [13, 8], [13, 9], [13, 10], [13, 11]]
right = [[6, 5], [6, 6], [7, 5], [7, 6], [7,7], [8, 5], [8,7], [9, 7], [9,6], [10,7], [14, 12], [14, 13], 
             [15, 12], [15, 14], [16, 12], [16, 13], [16, 14],[16,15]]

In [318]:
# Define channels in each region of interest
left_frontal = [[1, 1], [1, 2], [2, 1], [2, 2], [2, 3], [3, 1], [3, 3], [4, 3], [4, 2], [4, 4], [5, 3], [5, 4]]
left_parietal = [[11, 9], [11, 8], [12, 8], [12, 10], [13, 8], [13, 9], [13, 10], [13, 11]]

right_frontal = [[6, 5], [6, 6], [7, 5], [7, 6], [7,7], [8, 5], [8,7], [9, 7], [9,6], [10,7]] 
                 
right_parietal = [[14, 12], [14, 13],[15, 12], [15, 14], [16, 12], [16, 13], [16, 14],[16,15]]

In [336]:
rois = dict(
    # left_frontal=picks_pair_to_idx(raw_haemo, left_frontal),
    right_frontal=picks_pair_to_idx(raw_haemo, right_frontal),
    # left_parietal=picks_pair_to_idx(raw_haemo, left_parietal),
    right_parietal=picks_pair_to_idx(raw_haemo, right_parietal)
)

pprint(rois)

{'right_frontal': [13,
                   59,
                   14,
                   60,
                   15,
                   61,
                   16,
                   62,
                   17,
                   63,
                   18,
                   64,
                   19,
                   65,
                   21,
                   67,
                   20,
                   66,
                   22,
                   68],
 'right_parietal': [35,
                    81,
                    36,
                    82,
                    38,
                    84,
                    39,
                    85,
                    41,
                    87,
                    42,
                    88,
                    43,
                    89,
                    44,
                    90]}


In [338]:
# Specify the figure size and limits per chromophore.
fig, axes = plt.subplots(nrows=len(rois), ncols=len(all_evokeds), figsize=(15, 6))
lims = dict(hbo=[-3, 5], hbr=[-3, 5])

for pick, color in zip(["hbo", "hbr"], ["r", "b"]):
    for ridx, roi in enumerate(rois):
        for cidx, evoked in enumerate(all_evokeds):
            if pick == "hbr":
                picks = rois[roi][1::2]  # Select only the hbr channels
            else:
                picks = rois[roi][0::2]  # Select only the hbo channels

            plot_compare_evokeds(
                {evoked: all_evokeds[evoked]},
                combine="mean",
                picks=picks,
                axes=axes[ridx, cidx],
                show=False,
                colors=[color],
                legend=False,
                ylim=lims,
                ci=0.95,
                show_sensors=cidx == 1,
            )
            axes[0, cidx].set_title(f"{evoked}")
            axes[1, cidx].set_title("")
        axes[ridx, 0].set_ylabel(f"{roi}\nChromophore (ΔμMol)")